# Credit Card Fraud Detection
## 03 - Advanced Modeling: XGBoost, Threshold Optimization & Ensembling

### Objectives:
1. Standardize data preprocessing with duplicate removal and feature engineering (`Hour`).
2. Train and evaluate an **XGBoost Classifier** to capture complex non-linear patterns.
3. Perform **Precision-Recall Threshold Tuning** to balance False Positives and False Negatives.
4. Build and evaluate a **Soft-Voting Ensemble** combining complementary models.
5. Compare all candidate models side-by-side using imbalanced classification metrics (Precision, Recall, F1, ROC-AUC, PR-AUC).
6. Export serialized model artifacts for deployment in the Streamlit application.

In [1]:
# ============================================
# 1. IMPORTS & ENVIRONMENT SETUP
# ============================================
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

RANDOM_STATE = 42
pd.set_option('display.max_columns', None)
print('Libraries imported successfully!')

### 2. Data Loading & Preprocessing Pipeline
Following our findings in EDA and Notebook 02:
- Remove $1,081$ duplicate records.
- Extract `Hour` feature from `Time` ($(Time // 3600) \% 24$).
- Perform stratified 80/20 train/test split.
- Fit `StandardScaler` strictly on the training partition.

In [2]:
# Load raw data
df = pd.read_csv('../data/creditcard.csv')
print(f'Raw dataset shape: {df.shape}')

# 1. Deduplication
df = df.drop_duplicates().reset_index(drop=True)
print(f'Cleaned dataset shape after removing duplicates: {df.shape}')

# 2. Feature Engineering: Hour of Day
df['Hour'] = ((df['Time'] // 3600) % 24).astype(int)

# 3. Feature Matrix (X) and Target (y)
FEATURE_COLUMNS = [
    'Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9',
    'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19',
    'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Hour'
]
X = df[FEATURE_COLUMNS]
y = df['Class']

# 4. Stratified Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# 5. Standard Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training instances: {X_train.shape[0]:,} (Legitimate: {(y_train == 0).sum():,}, Fraud: {y_train.sum():,})')
print(f'Testing instances:  {X_test.shape[0]:,} (Legitimate: {(y_test == 0).sum():,}, Fraud: {y_test.sum():,})')

### 3. Evaluation Helper Function
We define a reusable function to capture key performance indicators with special focus on the minority (Fraud) class.

In [3]:
def evaluate_model(y_true, y_pred, y_prob, model_name):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    roc = roc_auc_score(y_true, y_prob)
    pr = average_precision_score(y_true, y_prob)
    
    print(f'========== {model_name.upper()} ==========')
    print(f'Precision : {p:.4f}')
    print(f'Recall    : {r:.4f}')
    print(f'F1-Score  : {f1:.4f}')
    print(f'ROC-AUC   : {roc:.4f}')
    print(f'PR-AUC    : {pr:.4f}')
    print(f'Confusion Matrix: TP={tp}, FP={fp}, FN={fn}, TN={tn}\n')
    
    return {
        'Model': model_name,
        'Precision': round(p, 4),
        'Recall': round(r, 4),
        'F1-Score': round(f1, 4),
        'ROC-AUC': round(roc, 4),
        'PR-AUC': round(pr, 4),
        'TP': int(tp),
        'FP': int(fp),
        'TN': int(tn),
        'FN': int(fn)
    }

### 4. Baseline & Resampling Models Review
We re-verify the Baseline Logistic Regression, Class-Weighted Logistic Regression, and SMOTE + Logistic Regression models on the scaled dataset.

In [4]:
model_results = []

# --- 1. Baseline Logistic Regression ---
lr_base = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_base.fit(X_train_scaled, y_train)
y_prob_base = lr_base.predict_proba(X_test_scaled)[:, 1]
y_pred_base = (y_prob_base >= 0.50).astype(int)
model_results.append(evaluate_model(y_test, y_pred_base, y_prob_base, 'Baseline Logistic Regression'))

# --- 2. Class-Weighted Logistic Regression ---
lr_weighted = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)
lr_weighted.fit(X_train_scaled, y_train)
y_prob_weighted = lr_weighted.predict_proba(X_test_scaled)[:, 1]
y_pred_weighted = (y_prob_weighted >= 0.50).astype(int)
model_results.append(evaluate_model(y_test, y_pred_weighted, y_prob_weighted, 'Class-Weighted Logistic Regression'))

# --- 3. SMOTE + Logistic Regression ---
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
lr_smote = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
lr_smote.fit(X_train_smote, y_train_smote)
y_prob_smote = lr_smote.predict_proba(X_test_scaled)[:, 1]
y_pred_smote = (y_prob_smote >= 0.50).astype(int)
model_results.append(evaluate_model(y_test, y_pred_smote, y_prob_smote, 'SMOTE + Logistic Regression'))

### 5. Advanced Tree-Based Model: XGBoost Classifier

Gradient boosted decision trees like **XGBoost** excel at finding non-linear decision boundaries and feature interactions without requiring synthetic sample generation.

**Key Hyperparameters:**
- `n_estimators=200`: Number of boosting iterations.
- `max_depth=5`: Depth of trees to capture interactions without overfitting.
- `learning_rate=0.08`: Shrinkage step size.
- `subsample=0.8` & `colsample_bytree=0.8`: Regularization via row and feature subsampling.
- `eval_metric='logloss'`: Binary logloss optimization.

In [5]:
# Instantiate and train XGBoost
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    eval_metric='logloss'
)

print('Training XGBoost Classifier...')
xgb.fit(X_train_scaled, y_train)
print('XGBoost training completed.')

# Predictions & Probabilities
y_prob_xgb = xgb.predict_proba(X_test_scaled)[:, 1]
y_pred_xgb_default = (y_prob_xgb >= 0.50).astype(int)

# Evaluate default XGBoost
model_results.append(evaluate_model(y_test, y_pred_xgb_default, y_prob_xgb, 'XGBoost Classifier (Threshold=0.50)'))

### 6. Threshold Optimization for XGBoost

In extreme fraud detection scenarios, standard threshold $0.50$ is not necessarily optimal. We scan probability thresholds from $0.05$ to $0.95$ to analyze the Precision/Recall/F1-Score trade-off curve.

In [6]:
# Threshold scan
thresholds = np.arange(0.05, 0.96, 0.02)
tuning_records = []

for t in thresholds:
    y_p = (y_prob_xgb >= t).astype(int)
    tuning_records.append({
        'Threshold': round(float(t), 2),
        'Precision': precision_score(y_test, y_p, zero_division=0),
        'Recall': recall_score(y_test, y_p, zero_division=0),
        'F1-Score': f1_score(y_test, y_p, zero_division=0)
    })

df_tuning = pd.DataFrame(tuning_records)
best_thresh_row = df_tuning.loc[df_tuning['F1-Score'].idxmax()]
optimal_threshold = float(best_thresh_row['Threshold'])

print(f'Optimal Threshold based on F1-Score: {optimal_threshold:.2f}')
print(best_thresh_row.to_dict())

# Plot Precision, Recall, F1 vs Threshold
plt.figure(figsize=(9, 5))
plt.plot(df_tuning['Threshold'], df_tuning['Precision'], marker='o', label='Precision', color='#1f77b4')
plt.plot(df_tuning['Threshold'], df_tuning['Recall'], marker='s', label='Recall', color='#2ca02c')
plt.plot(df_tuning['Threshold'], df_tuning['F1-Score'], marker='^', label='F1-Score', color='#d62728', linewidth=2)
plt.axvline(x=optimal_threshold, color='black', linestyle='--', label=f'Optimal Threshold = {optimal_threshold:.2f}')
plt.xlabel('Classification Threshold')
plt.ylabel('Score')
plt.title('XGBoost: Metric Trade-offs across Probability Thresholds')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Evaluate XGBoost at optimal threshold
y_pred_xgb_tuned = (y_prob_xgb >= optimal_threshold).astype(int)
model_results.append(evaluate_model(y_test, y_pred_xgb_tuned, y_prob_xgb, f'XGBoost (Tuned Thresh={optimal_threshold:.2f})'))

### 7. Soft-Voting Probability Ensemble

We build a soft-voting ensemble combining:
1. **XGBoost Classifier** (Weight = 0.85): Captures non-linear feature interactions.
2. **Baseline Logistic Regression** (Weight = 0.15): Linear regularized probability calibrator.

$$P_{\text{Ensemble}}(y=1) = 0.85 \cdot P_{\text{XGB}}(y=1) + 0.15 \cdot P_{\text{LR}}(y=1)$$

In [7]:
# Compute weighted ensemble probabilities
y_prob_ensemble = (0.85 * y_prob_xgb) + (0.15 * y_prob_base)
y_pred_ensemble_default = (y_prob_ensemble >= 0.50).astype(int)
y_pred_ensemble_tuned = (y_prob_ensemble >= optimal_threshold).astype(int)

model_results.append(evaluate_model(y_test, y_pred_ensemble_default, y_prob_ensemble, 'Soft-Voting Ensemble (Threshold=0.50)'))
model_results.append(evaluate_model(y_test, y_pred_ensemble_tuned, y_prob_ensemble, f'Soft-Voting Ensemble (Tuned Thresh={optimal_threshold:.2f})'))

### 8. Comprehensive Model Comparison & Analysis
We summarize the performance of all implemented models on the untouched test partition.

In [8]:
df_comparison = pd.DataFrame(model_results)
print('==================================== MODEL COMPARISON ====================================')
print(df_comparison.to_string(index=False))

# Save comparison to reports
os.makedirs('../reports', exist_ok=True)
df_comparison.to_csv('../reports/model_comparison.csv', index=False)

### 9. Feature Importance Analysis
Examine the top features driving fraud detection decisions in XGBoost.

In [9]:
importances = xgb.feature_importances_
df_feat = pd.DataFrame({'Feature': FEATURE_COLUMNS, 'Importance': importances}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=df_feat.head(12), color='#3182ce')
plt.title('Top 12 Most Influential Features in XGBoost Fraud Detection')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### 10. Save Model Artifacts for Streamlit Deployment
Serialize the champion model, fitted scaler, and operational configuration to `models/`.

In [10]:
os.makedirs('../models', exist_ok=True)

# Save Champion Model (XGBoost) and Scaler
joblib.dump(xgb, '../models/final_model.joblib')
joblib.dump(scaler, '../models/scaler.joblib')

# Save configuration metadata
config = {
    'model_name': 'XGBoost Classifier',
    'selected_threshold': optimal_threshold,
    'feature_names': FEATURE_COLUMNS,
    'random_state': RANDOM_STATE,
    'metrics': df_comparison.iloc[df_comparison['F1-Score'].idxmax()].to_dict()
}

with open('../models/model_config.json', 'w') as f:
    json.dump(config, f, indent=4)

print('All model artifacts saved successfully in models/!')